# Autoencoder on Galaxy10

We're replaying a classic experiment from **Hinton & Salakhutdinov, *Science* 2006** — ["Reducing the Dimensionality of Data with Neural Networks"](https://www.science.org/doi/10.1126/science.1127647). They trained a deep MLP autoencoder on MNIST and showed that the learned low-dimensional codes cleanly separated the digit classes, even though the model never saw labels during training. It was a pivotal demonstration that deep networks learn *structured representations*, not just input-output mappings.

We're going to replay that experiment with an astronomical twist. Instead of MNIST digits, we'll use **Galaxy10 SDSS**, a dataset of ~22k galaxy thumbnails labeled with 10 morphological classes. Same recipe — MLP encoder + bottleneck + MLP decoder, trained end-to-end with MSE — same question: **do morphologically similar galaxies end up near each other in the latent space?**

Concretely the autoencoder learns:
- an **encoder** $f: \mathbb{R}^{69 \times 69 \times 3} \to \mathbb{R}^{10}$ that compresses each image to a 10-dimensional code,
- a **decoder** $g: \mathbb{R}^{10} \to \mathbb{R}^{69 \times 69 \times 3}$ that reconstructs from the code.

Three acts:

1. **Warm-up** — gradient descent visualized in 2D, then a 1D MLP fit, to ground the PyTorch training loop.
2. **Train the autoencoder** — fit the bottleneck, look at reconstructions, visualize the latent space with PCA and t-SNE.
3. **Does the pretraining help?** — compare a classifier warm-started from the autoencoder's encoder against the same architecture trained from scratch. Faster convergence or higher final accuracy from the warm-started one would show the unsupervised pretraining transferred.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

## 1. Warm-up

Two warm-ups before the autoencoder. First we'll watch gradient descent on a 2-parameter problem so we can actually *see* the loss surface and the optimizer's path. Then we'll run the same pipeline on a 1D function with a small MLP.

In both, knobs sit at the top of each code cell — **re-run with different values** and see what changes.

### 1.1 Gradient descent, visualized

With only two parameters, the loss surface is a 2D function we can plot directly, and the optimizer's trajectory is a path on that surface. Here we fit a line $y = a\,x + b$ to noisy linear data, evaluate the MSE loss on a grid of $(a, b)$ values, and run SGD starting from a chosen initialization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Knobs ---
init_a, init_b = 0.0, 0.0    # starting point in (a, b) space
lr_2d          = 0.5
n_steps_2d     = 50
# ---

# Noisy linear data: true (a, b) = (3.0, 0.5)
torch.manual_seed(0)
true_a, true_b = 3.0, 0.5
x2d = torch.linspace(0, 1, 64, device=device).unsqueeze(1)
y2d = true_a * x2d + true_b + 0.2 * torch.randn_like(x2d)

def line_loss(a, b):
    return ((a * x2d + b - y2d) ** 2).mean()

# Loss on a (a, b) grid — vectorized in closed form for speed.
mean_x  = x2d.mean(); mean_x2 = (x2d ** 2).mean()
mean_y  = y2d.mean(); mean_y2 = (y2d ** 2).mean()
mean_xy = (x2d * y2d).mean()

a_grid = torch.linspace(-1, 6, 100, device=device)
b_grid = torch.linspace(-2, 3, 100, device=device)
A, B = torch.meshgrid(a_grid, b_grid, indexing="ij")
loss_grid = (A ** 2 * mean_x2 + 2 * A * B * mean_x - 2 * A * mean_xy
             + B ** 2 - 2 * B * mean_y + mean_y2)

# SGD trajectory on (a, b).
a = torch.tensor(init_a, device=device, requires_grad=True)
b = torch.tensor(init_b, device=device, requires_grad=True)
opt_2d = torch.optim.SGD([a, b], lr=lr_2d)

traj = [(a.item(), b.item())]
for step in range(n_steps_2d):
    opt_2d.zero_grad()
    line_loss(a, b).backward()
    opt_2d.step()
    traj.append((a.item(), b.item()))
traj = np.array(traj)

print(f"Final (a, b) = ({a.item():.3f}, {b.item():.3f}); true = ({true_a}, {true_b})")

In [ ]:
A_cpu   = A.cpu().numpy()
B_cpu   = B.cpu().numpy()
logloss = loss_grid.cpu().log().numpy()

fig, ax = plt.subplots(figsize=(8, 6))
cf = ax.contourf(A_cpu, B_cpu, logloss, levels=30, cmap="viridis")
fig.colorbar(cf, ax=ax, label="log MSE loss")
ax.contour(A_cpu, B_cpu, logloss, levels=10, colors="white", alpha=0.3, linewidths=0.5)
ax.plot(traj[:, 0], traj[:, 1], "o-", color="red", markersize=3, lw=1, label="SGD trajectory")
ax.plot(traj[0, 0], traj[0, 1], "*", color="yellow", markersize=18, label="start", zorder=5)
ax.plot(true_a, true_b, "+", color="white", markersize=15, mew=2, label="true min", zorder=5)
ax.set_xlabel("a (slope)"); ax.set_ylabel("b (intercept)")
ax.set_title(f"Loss landscape and SGD trajectory ({n_steps_2d} steps, lr={lr_2d})")
ax.legend()
plt.tight_layout()
plt.show()

#### Try these

Re-run the setup cell with these tweaks and watch the trajectory change:

- **Different start.** Try `init_a, init_b = 5, -1.5` or `(-1, 2.5)`. The bowl is convex, so SGD still finds the same minimum — but the path looks very different.
- **Bigger learning rate.** Set `lr_2d = 1.5`. Does the trajectory zig-zag across the valley before settling?
- **Way too big.** Set `lr_2d = 3.0`. Does SGD overshoot and diverge?
- **Tiny learning rate.** Set `lr_2d = 0.05`. How many steps does it take now to reach the bottom?

Notice the valley is elongated — the loss is much more sensitive to `a` than `b`. A single fixed step size has to be small enough not to overshoot along the steep direction, which makes progress along the shallow one painfully slow. This is exactly the problem **adaptive optimizers** like Adam (what we'll use for the autoencoder later) are designed to fix — they pick a per-parameter step size automatically.

### 1.2 The PyTorch pipeline on a 1D function

Now scale up: same loop (`zero_grad → forward → loss → backward → step`), but the model is a small MLP instead of two scalars, and we use Adam instead of SGD. The loss surface lives in hundreds of dimensions — we can't plot it anymore — but the principle is the same.

In [ ]:
# --- Target function ---
def target_fn(x):
    return torch.sin(2 * np.pi * x) + 0.3 * torch.cos(6 * np.pi * x)
# ---

n_samples = 256
x = torch.linspace(0, 1, n_samples, device=device).unsqueeze(1)
y = target_fn(x) + 0.05 * torch.randn_like(x)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x.cpu(), y.cpu(), ".", alpha=0.5, label="data")
ax.plot(x.cpu(), target_fn(x).cpu(), "-", label="truth")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Model and training knobs ---
hidden_width  = 32
hidden_layers = 3
lr            = 1e-2
n_steps       = 1000
# ---

# A tiny MLP, defined inline so the architecture is fully visible.
layers = [nn.Linear(1, hidden_width), nn.ReLU()]
for _ in range(hidden_layers - 1):
    layers += [nn.Linear(hidden_width, hidden_width), nn.ReLU()]
layers += [nn.Linear(hidden_width, 1)]
warmup_model = nn.Sequential(*layers).to(device)

opt_warmup = torch.optim.Adam(warmup_model.parameters(), lr=lr)

# Snapshot the prediction at initialization for the before/after plot.
with torch.no_grad():
    y_pred_init = warmup_model(x).cpu()

losses = []
for step in range(n_steps):
    opt_warmup.zero_grad()
    y_pred = warmup_model(x)
    loss = ((y_pred - y) ** 2).mean()
    loss.backward()
    opt_warmup.step()
    losses.append(loss.item())
    if (step + 1) % (n_steps // 10) == 0:
        print(f"step {step + 1:>5} | loss {loss.item():.4f}")

In [ ]:
with torch.no_grad():
    y_pred_final = warmup_model(x).cpu()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x.cpu(), y.cpu(), ".", alpha=0.4, label="data")
axes[0].plot(x.cpu(), y_pred_init,  "-", label="prediction (init)")
axes[0].plot(x.cpu(), y_pred_final, "-", label="prediction (trained)")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[0].set_title("Fit"); axes[0].legend()

axes[1].semilogy(losses)
axes[1].set_xlabel("step"); axes[1].set_ylabel("MSE loss")
axes[1].set_title("Training loss")

plt.tight_layout()
plt.show()

#### Try these

Re-run the training cell with each tweak and watch what changes:

- **Bigger model.** Set `hidden_width = 128` or `hidden_layers = 6`. Does the fit get visibly better?
- **Smaller model.** Set `hidden_width = 4`. Can the MLP represent the wiggly target at all?
- **Higher learning rate.** Set `lr = 1e-1`. Does training still converge, or does the loss blow up?
- **More steps.** Set `n_steps = 5000`. Does the loss keep decreasing or plateau?
- **Harder target.** Add a higher-frequency term to `target_fn`, e.g. `+ 0.2 * torch.sin(20 * np.pi * x)`. Does the MLP capture it?

That last one shows a limitation worth knowing about: **MLPs struggle with high-frequency content** without special architectural help. The autoencoder we'll train next doesn't fight that limitation — it just needs to compress and reconstruct, and *somewhat blurry* is fine for that. Sharpness isn't the point; representation structure is.

## 2. The Galaxy10 dataset

Galaxy10 SDSS is a curated set of ~22k galaxy thumbnails with morphology labels (smooth, spiral, edge-on, merging, etc.). Class counts are heavily imbalanced — the rarest class has only 17 examples. We use 90% for training and hold out 10% for validation.

In [ ]:
from tqdm import tqdm
from galaxy_dataset import load_galaxy10, CLASS_NAMES

data = load_galaxy10(device=device)
print(f"train: {data.train_images.shape[0]:>6,} images "
      f"(shape per image: {tuple(data.train_images.shape[1:])})")
print(f"val:   {data.val_images.shape[0]:>6,} images")

In [ ]:
# One example from each class, for a feel of what the model has to represent.
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for cls in range(len(CLASS_NAMES)):
    matches = (data.train_labels == cls).nonzero(as_tuple=True)[0]
    if matches.numel() == 0:
        axes.flat[cls].axis("off")
        continue
    img = (data.train_images[matches[0]].cpu().numpy() + 1) / 2   # [-1,1] -> [0,1] for display
    ax = axes.flat[cls]
    ax.imshow(img)
    ax.set_title(f"{cls}: {CLASS_NAMES[cls]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. The autoencoder

A pair of MLPs:

- **Encoder**: flatten the image to 14,283 features, pass through Linear+ReLU blocks (widths `(512, 128, 32)`), then a final linear projection to the 10-d latent vector.
- **Decoder**: mirror — linear up from 10 to 32 to 128 to 512 to 14,283, followed by `tanh` (matching the target range $[-1, 1]$) and reshape back to $(69, 69, 3)$.

No latent regularization, no skip connections — just a plain bottleneck autoencoder.

In [ ]:
from autoencoder import AutoencoderConfig, build_autoencoder

config = AutoencoderConfig(latent_dim=10, hidden_widths=(512, 128, 32))
encoder, decoder = build_autoencoder(config)
encoder.to(device); decoder.to(device)

n_enc = sum(p.numel() for p in encoder.parameters())
n_dec = sum(p.numel() for p in decoder.parameters())
print(f"encoder: {n_enc:>10,} params")
print(f"decoder: {n_dec:>10,} params")
print(f"total:   {n_enc + n_dec:>10,} params")

## 4. Training

Random shuffled mini-batches per epoch, MSE between input and reconstruction, Adam. Validation MSE is computed over the full held-out set after each epoch.

In [ ]:
# --- Knobs ---
batch_size = 256
n_epochs   = 30
ae_lr      = 1e-3
# ---

optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=ae_lr)
criterion = nn.MSELoss()

n_train = data.train_images.shape[0]
train_losses, val_losses = [], []

pbar = tqdm(range(n_epochs), desc="Training")
for epoch in pbar:
    perm = torch.randperm(n_train, device=device)
    epoch_loss, n_batches = 0.0, 0
    for start in range(0, n_train, batch_size):
        idx = perm[start:start + batch_size]
        batch = data.train_images[idx]
        recon = decoder(encoder(batch))
        loss = criterion(recon, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    train_losses.append(epoch_loss / n_batches)

    with torch.no_grad():
        val_loss = criterion(decoder(encoder(data.val_images)), data.val_images).item()
    val_losses.append(val_loss)

    pbar.set_postfix(train=f"{train_losses[-1]:.4f}", val=f"{val_loss:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, "-o", label="train", markersize=4)
ax.plot(val_losses,   "-o", label="val",   markersize=4)
ax.set_xlabel("epoch"); ax.set_ylabel("MSE loss")
ax.set_yscale("log")
ax.set_title(f"Autoencoder training — latent_dim={config.latent_dim}")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Reconstruction quality

Top row: original validation images. Bottom row: reconstructions through the 10-d bottleneck. They won't be pixel-perfect — that's expected at a compression ratio of ~14283 / 10 ≈ 1400× — but the morphology should be recognizable.

In [ ]:
n_show = 8
with torch.no_grad():
    sample = data.val_images[:n_show]
    recon  = decoder(encoder(sample))
orig_np  = ((sample.cpu().numpy() + 1) / 2).clip(0, 1)
recon_np = ((recon.cpu().numpy()  + 1) / 2).clip(0, 1)

fig, axes = plt.subplots(2, n_show, figsize=(2 * n_show, 4.5))
for i in range(n_show):
    axes[0, i].imshow(orig_np[i]);  axes[0, i].axis("off")
    axes[1, i].imshow(recon_np[i]); axes[1, i].axis("off")
fig.text(0.01, 0.75, "original",       rotation=90, va="center", fontsize=11)
fig.text(0.01, 0.27, "reconstruction", rotation=90, va="center", fontsize=11)
plt.tight_layout(rect=(0.03, 0, 1, 1))
plt.show()

## 6. The latent space

This is the payoff cell — the Hinton & Salakhutdinov moment, restaged on galaxies. Encode every training image to a 10-d code, then project to 2D two ways:

- **PCA** (linear): the two directions of maximum variance in latent space. Honest about distances, but if class structure isn't aligned with the top variance directions it'll wash out.
- **t-SNE** (nonlinear): preserves *local neighborhoods* — points that are nearby in 10-d stay nearby in 2-d. Distances and cluster *sizes* aren't meaningful, but cluster *membership* usually is. Stochastic.

> **Downsampling for speed.** t-SNE on all ~20k training points takes ~30 s. For the demo we instead pick **up to 100 samples per class** (~1000 points total), which gives the same qualitative picture in a couple of seconds. Rare classes (e.g. class 5 with 17 examples total) contribute everything they have. Both PCA and t-SNE are run on this subset so the two panels are directly comparable.

Color each point by its true class label — the autoencoder never saw labels during training, so any clustering by morphology is **emergent structure** in the latent space. If morphologically similar galaxies (e.g. all the edge-on classes) land near each other in t-SNE, the autoencoder has learned a representation that aligns with how humans categorize galaxies.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Encode all training images to z-space.
with torch.no_grad():
    z = encoder(data.train_images).cpu().numpy()    # (N_train, 16)
labels_np = data.train_labels.cpu().numpy()

# Downsample: up to 100 samples per class. t-SNE on the full ~20k points takes
# ~30s; ~1000 points takes ~3s and gives the same qualitative picture.
samples_per_class = 100
rng = np.random.default_rng(0)
subset_idx = np.concatenate([
    rng.choice(np.where(labels_np == cls)[0],
               size=min(samples_per_class, int((labels_np == cls).sum())),
               replace=False)
    for cls in range(len(CLASS_NAMES))
])
z_subset      = z[subset_idx]
labels_subset = labels_np[subset_idx]
print(f"Visualizing {z_subset.shape[0]} points "
      f"(downsampled from {z.shape[0]}, up to {samples_per_class}/class)")

# PCA: linear, exact, fast.
pca = PCA(n_components=2)
z_pca = pca.fit_transform(z_subset)
print(f"PCA explains {pca.explained_variance_ratio_.sum():.1%} of subset latent variance")

# t-SNE: nonlinear, stochastic. init='pca' + learning_rate='auto' are the
# modern defaults that avoid the bad-local-minima failure modes of older t-SNE.
print("Running t-SNE...")
z_tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=0,
).fit_transform(z_subset)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, embed, name in [(axes[0], z_pca, "PCA"), (axes[1], z_tsne, "t-SNE")]:
    for cls in range(len(CLASS_NAMES)):
        mask = labels_subset == cls
        ax.scatter(embed[mask, 0], embed[mask, 1], s=12, alpha=0.7,
                   label=f"{cls}: {CLASS_NAMES[cls]} (n={mask.sum()})")
    ax.set_xlabel(f"{name} 1"); ax.set_ylabel(f"{name} 2")
    ax.set_title(f"Latent space — {name}")

axes[1].legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=9, markerscale=2)
plt.tight_layout()
plt.show()

## 7. Does the autoencoder pretraining actually help?

The autoencoder learned a representation without ever seeing class labels. Does that representation help a *supervised* classifier do its job better than starting from scratch? Compare two end-to-end setups:

- **(B) Pretrained init, fine-tuned:** copy the autoencoder's encoder weights, attach a fresh `Linear(latent_dim, 10)` head, train *everything* end-to-end with cross-entropy.
- **(C) From scratch:** identical architecture (encoder + head), random init, same training loop.

If B beats C — or just converges faster — the unsupervised pretraining gave a useful warm start. If they end at the same final accuracy, the encoder architecture (not the pretraining) was doing all the work.

In [ ]:
# --- Knobs ---
ft_steps = 1000
ft_lr    = 1e-3
ft_batch = 256
# ---

ce_loss = nn.CrossEntropyLoss()

# Snapshot the trained encoder weights so this section can re-run without re-training the AE.
encoder_pretrained_state = {k: v.clone() for k, v in encoder.state_dict().items()}


def build_full_classifier(init_from=None):
    """Encoder architecture + Linear(latent_dim, n_classes) head. If init_from
    is given, load those weights into the encoder before training."""
    enc, _ = build_autoencoder(config)
    if init_from is not None:
        enc.load_state_dict(init_from)
    head = nn.Linear(config.latent_dim, len(CLASS_NAMES))
    return nn.Sequential(enc, head).to(device)


def train_end_to_end(model, n_steps, batch_size, lr, desc):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n_train = data.train_images.shape[0]
    train_log, val_log = [], []
    pbar = tqdm(range(n_steps), desc=desc)
    for step in pbar:
        idx = torch.randint(0, n_train, (batch_size,), device=device)
        logits = model(data.train_images[idx])
        loss = ce_loss(logits, data.train_labels[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        train_log.append((logits.argmax(1) == data.train_labels[idx]).float().mean().item())
        with torch.no_grad():
            val_log.append((model(data.val_images).argmax(1) == data.val_labels).float().mean().item())
        if (step + 1) % 50 == 0:
            pbar.set_postfix(loss=f"{loss.item():.3f}", val=f"{val_log[-1]:.3f}")
    return train_log, val_log


# (B) Pretrained init, fine-tuned end-to-end.
model_B = build_full_classifier(init_from=encoder_pretrained_state)
trainB_log, valB_log = train_end_to_end(model_B, ft_steps, ft_batch, ft_lr, "B: pretrained + ft")
print(f"Final val accuracy (B): {valB_log[-1]:.3f}")

In [ ]:
# (C) From scratch — same architecture, random init, same training loop.
model_C = build_full_classifier(init_from=None)
trainC_log, valC_log = train_end_to_end(model_C, ft_steps, ft_batch, ft_lr, "C: from scratch")
print(f"Final val accuracy (C): {valC_log[-1]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(valB_log, label=f"B: pretrained + fine-tune — final {valB_log[-1]:.3f}", color="C1")
ax.plot(valC_log, label=f"C: from scratch           — final {valC_log[-1]:.3f}", color="C2")
ax.set_xlabel("step")
ax.set_ylabel("validation accuracy")
ax.set_title("Galaxy10 classification: pretrained vs from scratch")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()